# Deploy Clinical Data Agent

This notebook builds the Genie configuration from the repository JSON files and deploys it to Databricks.

Deployment workflow:

1. Locate the repository deployment code
2. Build and validate the Genie configuration
3. Check whether the Clinical Data Agent already exists
4. Perform a dry run
5. Create or update the Genie space
6. Validate the deployed configuration

In [0]:
from pathlib import Path
import sys


# Locate repository root
current_path = Path.cwd().resolve()
repo_root = None

for candidate in [current_path] + list(current_path.parents):
    if (
        (candidate / "Configs").exists()
        and (candidate / "deployment" / "build_genie_config.py").exists()
    ):
        repo_root = candidate
        break


if repo_root is None:
    raise RuntimeError(
        "Could not locate the repository root."
    )


# Make deployment modules importable
deployment_dir = repo_root / "deployment"

if str(deployment_dir) not in sys.path:
    sys.path.insert(
        0,
        str(deployment_dir)
    )


print(f"Repository root: {repo_root}")
print(f"Deployment directory: {deployment_dir}")

In [0]:
# # Existing Genie Agent to update
# SPACE_ID = "01f1b4aab3d7116dbd4f02d504887741"

# # Safety check
# EXPECTED_TITLE = "Clinical Data Agent"

In [0]:
# Genie deployment settings
GENIE_TITLE = "Clinical Data Agent"

# Replace this with the SQL Warehouse ID
# from the Premium workspace before deployment

#WAREHOUSE_ID = "<PREMIUM_SQL_WAREHOUSE_ID>"
WAREHOUSE_ID = "ee49b89605fae98d"

GENIE_DESCRIPTION = (
    "Natural-language analytics for the "
    "Modern Clinical Data Repository."
)

# Optional workspace folder for the Genie space
PARENT_PATH = None

In [0]:
import importlib

import build_genie_config
import deploy_genie_agent


# Reload repository modules
importlib.reload(build_genie_config)
importlib.reload(deploy_genie_agent)


from build_genie_config import (
    build_genie_config,
    export_genie_config
)

from deploy_genie_agent import (
    deploy_genie_agent,
    validate_deployed_agent,
    find_genie_space_by_title
)


# Build and export a readable preview
config = build_genie_config()
output_file = export_genie_config()


sql_snippets = (
    config
    .get("instructions", {})
    .get("sql_snippets", {})
)


print("Genie configuration built successfully.")
print()
print(f"Version: {config['version']}")
print(
    f"Tables: "
    f"{len(config['data_sources']['tables'])}"
)
print(
    f"Sample questions: "
    f"{len(config['config']['sample_questions'])}"
)
print(
    f"SQL examples: "
    f"{len(config['instructions']['example_question_sqls'])}"
)
print(
    f"Join specs: "
    f"{len(config['instructions']['join_specs'])}"
)
print(
    f"SQL filters: "
    f"{len(sql_snippets.get('filters', []))}"
)
print(
    f"SQL expressions: "
    f"{len(sql_snippets.get('expressions', []))}"
)
print(
    f"SQL measures: "
    f"{len(sql_snippets.get('measures', []))}"
)
print(
    f"Benchmarks: "
    f"{len(config['benchmarks']['questions'])}"
)
print()
print(f"Generated preview: {output_file}")

In [0]:
dry_run_result = deploy_genie_agent(
    title=GENIE_TITLE,
    warehouse_id=WAREHOUSE_ID,
    description=GENIE_DESCRIPTION,
    parent_path=PARENT_PATH,
    dry_run=True
)

dry_run_result

In [0]:
# Check whether the Genie space already exists
existing_space = find_genie_space_by_title(
    GENIE_TITLE
)

if existing_space:
    print("Existing Genie space found.")
    print(
        f"Space ID: "
        f"{existing_space.get('space_id')}"
    )
    print(
        f"Title: "
        f"{existing_space.get('title')}"
    )
else:
    print(
        "No existing Genie space found. "
        "The first deployment will create one."
    )

In [0]:
deployment_result = deploy_genie_agent(
    title=GENIE_TITLE,
    warehouse_id=WAREHOUSE_ID,
    description=GENIE_DESCRIPTION,
    parent_path=PARENT_PATH,
    dry_run=False
)

deployment_result

In [0]:
# Use the Space ID returned directly by the deployment
SPACE_ID = deployment_result.get("space_id")

if not SPACE_ID:
    raise RuntimeError(
        "Deployment completed but no Space ID was returned."
    )

print(f"Deployed Space ID: {SPACE_ID}")


# Validate the deployed configuration
validation_summary = validate_deployed_agent(
    SPACE_ID
)

validation_summary